# Tarea 1: Entendimiento

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions
from pyspark.sql.types import StructType
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql.types import FloatType, StringType, IntegerType, DateType
from pyspark.sql.functions import udf, col, length, isnan, when, count
import pyspark.sql.functions as f
import os 
from datetime import datetime
from pyspark.sql import types as t
from pandas_profiling import ProfileReport
import matplotlib.pyplot as plt
import numpy as np

/home/vscode/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_31412/2355262383.py:11: DeprecationWarning: `import pandas_profiling` is going to be deprecated by April 1st. Please use `import ydata_profiling` instead.
  from pandas_profiling import ProfileReport


Configuración del controlador e inicio de sesion Spark

In [2]:
path_jar_driver = "/workspaces/ETL/drivers/mysql-connector-j.jar"

In [3]:
#Configuración de la sesión
conf = SparkConf() \
    .set("spark.driver.extraClassPath", path_jar_driver) \
    .set("spark.jars", path_jar_driver)

spark_context = SparkContext(conf=conf)
sql_context = SQLContext(spark_context)
spark = sql_context.sparkSession

Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
26/06/16 23:33:18 WARN Utils: Your hostname, codespaces-860f81 resolves to a loopback address: 127.0.0.1; using 10.0.10.233 instead (on interface eth0)
26/06/16 23:33:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/06/16 23:33:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/vscode/.local/lib/python3.11/site-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [4]:
# Si quiere practicar la conexion con el servidor de base de datos:
db_connection_string = 'jdbc:mysql://157.253.236.120:8080/WWImportersTransactional'
# El usuario es su estudiante _i asignado y su contraseña la encontrará en el archivo excel de Coursera 
db_user = 'DB_202613_dp_rojasr1'
db_psswd = '202116178'

PATH='./'

In [5]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd

In [6]:
print("db_user:", db_user)
print("db_psswd vacío?:", db_psswd == "" or db_psswd is None)

db_user: DB_202613_dp_rojasr1
db_psswd vacío?: False


Para comprobar su comprensión de esta sección, lo invitamos a responder la pregunta:

    ¿Qué funcionalidades de PySpark puedo utilizar para realizar el entendimiento de datos y cómo las puedo utilizar?
    
Como parte de nuestro trabajo es preveer posibles soluciones a las conclusiones de perfilamiento, por ende lo invitamos a responder:

    ¿Qué métodos podría utilizar para reemplazar los valores nulos de una columna por el promedio de la misma?

# 1. ¿Qué funcionalidades de PySpark puedo utilizar para realizar el entendimiento de datos y cómo las puedo utilizar?

Antes de construir indicadores, modelos analíticos o procesos de transformación, es necesario conocer las características de la información disponible. El entendimiento de datos permite identificar cómo están estructurados los datos, qué tan confiables son, qué problemas presentan y qué tan útiles pueden ser para responder las necesidades del negocio.

PySpark ofrece múltiples herramientas que facilitan este proceso incluso cuando se trabaja con grandes volúmenes de información.

## Conocer la estructura de los datos

Una de las primeras actividades consiste en revisar cómo está compuesto el conjunto de datos.

### printSchema()

Esta función permite visualizar el esquema del DataFrame, mostrando el nombre de cada columna, su tipo de dato y si admite valores nulos.

```python
df.printSchema()
```

Su principal utilidad es detectar inconsistencias entre el significado de un campo y el tipo de dato con el que fue almacenado. También permite identificar qué variables podrían requerir transformaciones antes de ser utilizadas.

### columns

Permite obtener la lista de atributos disponibles en el DataFrame.

```python
df.columns
```

Resulta útil para familiarizarse rápidamente con la estructura de la información y planificar qué variables serán utilizadas en los análisis posteriores.

---

## Explorar los registros almacenados

Después de revisar la estructura, es importante observar ejemplos reales de los datos.

### show()

Permite visualizar una muestra de registros.

```python
df.show(10)
```

Esta función ayuda a interpretar el significado de cada fila, detectar formatos inesperados, identificar posibles errores de captura y validar que la información fue cargada correctamente.

### count()

Calcula el número total de registros.

```python
df.count()
```

Es una métrica básica para dimensionar el volumen de información disponible y contrastarlo con las cifras reportadas por el negocio.

---

## Obtener estadísticas descriptivas

Las estadísticas descriptivas permiten comprender el comportamiento general de las variables numéricas.

### describe()

Genera métricas básicas como:

* Número de observaciones
* Promedio
* Desviación estándar
* Valor mínimo
* Valor máximo

```python
df.describe().show()
```

Estas medidas permiten identificar posibles valores extremos y entender el rango general de los datos.

### summary()

Amplía el análisis incorporando percentiles y medidas de posición.

```python
df.summary().show()
```

Los percentiles permiten comprender cómo se distribuyen los valores dentro de una variable y facilitan la detección de comportamientos atípicos.

---

## Analizar valores únicos

### distinct()

Permite identificar cuántos valores diferentes existen dentro de una columna.

```python
df.select("ProductoID").distinct().count()
```

Este análisis es útil para validar cardinalidades y verificar si la información coincide con lo esperado por el negocio.

Por ejemplo, puede utilizarse para conocer cuántos clientes, proveedores o productos aparecen realmente en los datos.

---

## Analizar frecuencias y distribuciones

### groupBy()

Permite agrupar registros y calcular métricas por categoría.

```python
df.groupBy("TipoTransaccionID").count().show()
```

Es especialmente útil para identificar categorías predominantes, detectar concentraciones de información o encontrar comportamientos poco frecuentes.

### agg()

Permite realizar múltiples cálculos agregados en una sola consulta.

```python
from pyspark.sql.functions import *

df.agg(
    count("*").alias("Registros"),
    avg("Cantidad").alias("Promedio"),
    max("Cantidad").alias("ValorMaximo")
).show()
```

Esta funcionalidad resulta muy útil para construir indicadores generales y validar reglas operativas.

---

## Validar reglas de negocio

Una parte importante del entendimiento consiste en verificar que los datos respeten las restricciones definidas por la organización.

### filter() o where()

Permiten seleccionar únicamente los registros que cumplen una condición determinada.

```python
df.filter(col("Cantidad") > 50000000).show()
```

A través de este tipo de consultas es posible detectar valores que incumplen límites establecidos por el negocio o identificar registros que requieren revisión.

---

## Identificar valores extremos

### orderBy()

Permite ordenar los registros de acuerdo con una o varias columnas.

```python
df.orderBy(col("Cantidad").desc()).show()
```

Esta función facilita la identificación de valores máximos y mínimos, así como la revisión de posibles anomalías.

---

## Evaluar la calidad de los datos

El entendimiento de datos también implica medir qué tan confiable es la información.

### Identificación de valores nulos

```python
df.filter(col("Cantidad").isNull()).count()
```

Permite determinar qué tan completa se encuentra una variable y medir la cantidad de información faltante.

### Identificación de registros duplicados

```python
df.dropDuplicates().count()
```

Ayuda a detectar problemas de duplicidad que podrían afectar indicadores o generar resultados incorrectos.

### Validación de formatos

Por ejemplo, en variables de fecha es posible validar si los registros cumplen con el formato esperado.

```python
from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "FechaMovimiento",
    to_timestamp("FechaMovimiento", "yyyy-MM-dd HH:mm:ss")
)
```

Este tipo de validaciones permite detectar inconsistencias antes de realizar análisis temporales.

---

## Conclusión

PySpark proporciona un conjunto de herramientas que permiten entender los datos desde diferentes perspectivas: estructura, volumen, distribución, calidad y cumplimiento de reglas de negocio. La combinación de estas funcionalidades facilita la identificación temprana de problemas, la formulación de preguntas para las áreas usuarias y la definición de estrategias de limpieza que mejoren la confiabilidad de los análisis posteriores.

# 2. ¿Qué métodos podría utilizar para reemplazar los valores nulos de una columna por el promedio de la misma?

Cuando una variable numérica presenta valores nulos, una alternativa común consiste en reemplazarlos por una medida representativa de la distribución. Una de las opciones más utilizadas es el promedio de la propia columna, ya que permite conservar los registros y evitar la pérdida de información durante el análisis.

## Alternativa 1: Utilizar fillna()

Primero se calcula el promedio de la variable:

```python
from pyspark.sql.functions import avg

promedio = df.select(
    avg("Cantidad")
).first()[0]
```

Posteriormente se reemplazan los valores nulos:

```python
df = df.fillna({
    "Cantidad": promedio
})
```

Esta alternativa es sencilla de implementar y suele utilizarse cuando la proporción de valores faltantes es relativamente baja.

---

## Alternativa 2: Utilizar withColumn() y when()

Otra posibilidad consiste en reemplazar los valores faltantes mediante una condición explícita.

```python
from pyspark.sql.functions import when, col

df = df.withColumn(
    "Cantidad",
    when(
        col("Cantidad").isNull(),
        promedio
    ).otherwise(col("Cantidad"))
)
```

Este enfoque ofrece mayor flexibilidad porque permite incorporar reglas adicionales dependiendo del contexto del negocio.

---

## Alternativa 3: Realizar la imputación mediante SQL

Cuando se trabaja con consultas SQL sobre Spark, la sustitución también puede realizarse directamente dentro de una consulta.

```sql
SELECT
    CASE
        WHEN Cantidad IS NULL THEN (
            SELECT AVG(Cantidad)
            FROM movimientos
        )
        ELSE Cantidad
    END AS Cantidad
FROM movimientos
```

Este método puede resultar conveniente cuando gran parte de las transformaciones se implementan mediante sentencias SQL.

---

## Aspectos a evaluar antes de imputar valores

Aunque reemplazar nulos por el promedio es una práctica frecuente, no siempre es la mejor decisión. Antes de hacerlo es recomendable analizar:

* Qué porcentaje de registros presenta valores faltantes.
* Si el valor nulo corresponde realmente a información ausente.
* El impacto que la imputación puede tener sobre las métricas del negocio.
* Si existen métodos más apropiados según la naturaleza de la variable.

En algunos escenarios puede ser más adecuado utilizar la mediana, realizar imputaciones segmentadas o incluso eliminar ciertos registros.

---

## Conclusión

PySpark permite implementar distintas estrategias para sustituir valores nulos utilizando funciones nativas del framework. La elección del método dependerá tanto de las características de los datos como del objetivo del análisis. Aunque reemplazar por el promedio suele ser una solución práctica y rápida, siempre debe evaluarse su impacto sobre la interpretación final de los resultados.


# 5. Tarea
Espacio para desarrollar la tarea propuesta 

### Perfilamiento de datos

In [7]:
sql_movimientos = "(SELECT * FROM movimientosCopia) movimientosCopia"

movimientos = obtener_dataframe_de_bd(
    db_connection_string,
    sql_movimientos,
    db_user,
    db_psswd
)

In [8]:
movimientos.printSchema()

movimientos.show(5)

print("Filas:", movimientos.count())

print("Columnas:", len(movimientos.columns))

root
 |-- TransaccionProductoID: integer (nullable = true)
 |-- ProductoID: integer (nullable = true)
 |-- TipoTransaccionID: integer (nullable = true)
 |-- ClienteID: double (nullable = true)
 |-- InvoiceID: double (nullable = true)
 |-- ProveedorID: string (nullable = true)
 |-- OrdenDeCompraID: string (nullable = true)
 |-- FechaTransaccion: string (nullable = true)
 |-- Cantidad: double (nullable = true)



+---------------------+----------+-----------------+---------+---------+-----------+---------------+----------------+--------+
|TransaccionProductoID|ProductoID|TipoTransaccionID|ClienteID|InvoiceID|ProveedorID|OrdenDeCompraID|FechaTransaccion|Cantidad|
+---------------------+----------+-----------------+---------+---------+-----------+---------------+----------------+--------+
|               118903|       217|               10|    476.0|  24904.0|           |               |     Apr 25,2014|   -40.0|
|               286890|       135|               10|     33.0|  60117.0|           |               |     Dec 10,2015|    -7.0|
|               285233|       111|               10|    180.0|  59768.0|           |               |     Dec 04,2015|    -2.0|
|               290145|       213|               10|     33.0|  60795.0|           |               |     Dec 23,2015|    -3.0|
|               247492|        90|               10|     55.0|  51851.0|           |               |     Jul 27

Filas: 204292
Columnas: 9


## Significado de una fila

Al revisar los campos disponibles en la tabla `movimientosCopia`, se puede interpretar que cada registro corresponde a un evento específico que genera una modificación en el inventario de un producto.

La fila identifica qué producto fue afectado, cuándo ocurrió el movimiento, qué tipo de transacción lo originó y qué entidades participaron en la operación. Dependiendo del contexto, el movimiento puede estar relacionado con un cliente, una factura, un proveedor o una orden de compra.

Adicionalmente, el campo `Cantidad` permite medir el impacto del movimiento sobre las existencias del producto. Por esta razón, cada registro puede entenderse como una evidencia histórica de una entrada o una salida de inventario ocurrida dentro de la operación de la compañía.

En conjunto, los registros de esta tabla permiten reconstruir la trazabilidad de los movimientos realizados sobre los productos y analizar cómo ha evolucionado el inventario a lo largo del tiempo.

## Estructura de los datos

La tabla `movimientosCopia` contiene 9 atributos que describen diferentes características de los movimientos de inventario registrados por la organización.

| Campo                 | Tipo de dato |
| --------------------- | ------------ |
| TransaccionProductoID | Integer      |
| ProductoID            | Integer      |
| TipoTransaccionID     | Integer      |
| ClienteID             | Double       |
| InvoiceID             | Double       |
| ProveedorID           | String       |
| OrdenDeCompraID       | String       |
| FechaTransaccion      | String       |
| Cantidad              | Double       |

A partir de la revisión del esquema se observa que la mayoría de los atributos corresponden a identificadores utilizados para relacionar el movimiento con otros elementos del proceso de negocio, tales como productos, clientes, proveedores, facturas u órdenes de compra.

La variable `Cantidad` representa la magnitud del movimiento realizado sobre el inventario y constituye uno de los atributos más relevantes para los análisis posteriores.

Por otra parte, `FechaTransaccion` almacena la información temporal asociada al movimiento. Aunque conceptualmente corresponde a una fecha, actualmente se encuentra almacenada como texto, situación que deberá revisarse durante el análisis de calidad de datos para determinar si el formato utilizado es consistente y adecuado para realizar consultas temporales.

En términos generales, la tabla combina variables numéricas, atributos de identificación y campos de carácter temporal, proporcionando la información mínima necesaria para analizar la dinámica de los movimientos de inventario registrados por Wide World Importers.


In [9]:
movimientos.describe().show()

26/06/16 23:33:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+---------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+--------------------+-----------------+
|summary|TransaccionProductoID|        ProductoID|  TipoTransaccionID|         ClienteID|        InvoiceID|      ProveedorID|   OrdenDeCompraID|    FechaTransaccion|         Cantidad|
+-------+---------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+--------------------+-----------------+
|  count|               204292|            204292|             204292|            204292|           204292|           204292|            204292|              204292|           204292|
|   mean|   212458.04047637695|110.70090850351458| 10.035253460732676| 517.3252941867523|42957.26929590978|4.951898734177215|1345.9973277074544|                NULL|719.4997650421946|
| stddev|    71352.37579752573| 63.49014746219581|0.18563716955046372|353.501369

## Estadísticas descriptivas

Con el objetivo de comprender el comportamiento general de los movimientos de inventario, se realizó una exploración estadística de las variables numéricas presentes en la tabla `movimientosCopia`. Este análisis permite identificar rangos de valores, dispersión de los datos y posibles comportamientos atípicos que deberán considerarse en etapas posteriores.

### Identificadores de productos

Los movimientos registrados abarcan productos con identificadores comprendidos entre 1 y 227. Este resultado coincide con la información observada durante la exploración de los datos, donde se identificaron 227 productos distintos con al menos un movimiento de inventario registrado.

La amplitud de productos involucrados sugiere que la información disponible cubre una porción importante del catálogo administrado por la compañía.

### Tipos de transacción

Los registros se encuentran asociados a tres tipos de transacción diferentes, representados por los identificadores 10, 11 y 12.

Aunque el significado específico de cada código no se encuentra documentado en la información suministrada, la existencia de únicamente tres categorías indica que la operación de inventario se concentra en un conjunto reducido de eventos de negocio claramente definidos.

### Clientes

Los identificadores de clientes presentan valores entre 0 y 1.061.

La presencia del valor 0 resulta llamativa, ya que podría representar movimientos que no se encuentran asociados a un cliente específico o registros donde la información no fue diligenciada. Este comportamiento deberá analizarse con mayor detalle durante la evaluación de calidad de datos.

### Facturas

Los identificadores de factura oscilan entre 0 y 70.510.

Al igual que ocurre con los clientes, la existencia de registros con valor 0 podría indicar operaciones que no cuentan con una factura asociada o situaciones especiales dentro del proceso de negocio que requieren validación con los responsables de la información.

### Proveedores

Los movimientos contienen referencias a proveedores identificados principalmente mediante los códigos 1, 4 y 7. Adicionalmente, una gran cantidad de registros no presenta información de proveedor.

Este comportamiento sugiere que la participación de proveedores depende del tipo de transacción realizada y que no todos los movimientos requieren registrar esta información.

### Órdenes de compra

Los identificadores de órdenes de compra presentan valores entre 999 y 1.345.

Sin embargo, durante la exploración se observó que un número considerable de registros no tiene una orden de compra asociada, situación que parece estar relacionada con la naturaleza de determinadas transacciones de inventario.

### Cantidad de productos movidos

La variable `Cantidad` representa el volumen de unidades afectadas por cada movimiento de inventario.

Los principales indicadores obtenidos fueron:

* Valor mínimo: -360 unidades
* Percentil 25: -60 unidades
* Mediana: -9 unidades
* Percentil 75: -5 unidades
* Valor máximo: 67.368 unidades
* Promedio: 719 unidades
* Desviación estándar: 4.729 unidades

Los resultados muestran una distribución altamente dispersa. La diferencia entre la mediana (-9) y el promedio (719) evidencia la presencia de movimientos con cantidades excepcionalmente altas que desplazan el promedio hacia valores positivos.

Adicionalmente, la mayoría de los registros presentan cantidades negativas, lo que sugiere que gran parte de los movimientos corresponden a salidas de inventario. Este comportamiento será validado posteriormente mediante el análisis de reglas de negocio.

### Cobertura temporal

La información disponible cubre movimientos realizados entre diciembre de 2013 y septiembre de 2015.

Por lo tanto, el conjunto de datos permite analizar cerca de dos años de operación, proporcionando una base suficiente para estudiar tendencias, patrones de movimiento y comportamiento histórico del inventario.

### Conclusión

Las estadísticas descriptivas muestran una operación con una amplia diversidad de productos, clientes y documentos asociados. También evidencian una fuerte concentración de movimientos negativos y una alta dispersión en las cantidades registradas, aspectos que deberán ser considerados durante los análisis posteriores.

Asimismo, se identifican valores especiales como clientes o facturas con identificador cero y una elevada ausencia de información de proveedores y órdenes de compra, elementos que requieren validación con el negocio para determinar si corresponden a situaciones operativas esperadas o a posibles problemas de calidad de datos.


In [10]:
from pyspark.sql.functions import countDistinct

movimientos.select(
    countDistinct("ClienteID").alias("Clientes"),
    countDistinct("ProveedorID").alias("Proveedores"),
    countDistinct("ProductoID").alias("Productos"),
    countDistinct("TipoTransaccionID").alias("TiposTransaccion")
).show()

+--------+-----------+---------+----------------+
|Clientes|Proveedores|Productos|TiposTransaccion|
+--------+-----------+---------+----------------+
|     664|          4|      227|               3|
+--------+-----------+---------+----------------+



## Cardinalidad de las entidades

Como parte del entendimiento de los datos, se calculó la cantidad de valores distintos presentes en algunos de los atributos más relevantes de la tabla. Este análisis permite dimensionar el alcance de la información disponible y conocer el nivel de diversidad existente en los registros.

| Entidad              | Cantidad de valores distintos |
| -------------------- | ----------------------------: |
| Clientes             |                           664 |
| Proveedores          |                            4* |
| Productos            |                           227 |
| Tipos de transacción |                             3 |

*Uno de los valores identificados corresponde a registros donde el campo `ProveedorID` se encuentra vacío.

### Clientes

Se identificaron 664 clientes diferentes con al menos un movimiento asociado durante el período analizado. Esto indica que la actividad registrada involucra una base relativamente amplia de clientes y sugiere que los movimientos de inventario no se concentran en un grupo reducido de compradores.

### Proveedores

Inicialmente se detectaron cuatro valores distintos para el atributo `ProveedorID`. Sin embargo, al revisar el detalle de los registros se observó que uno de estos valores corresponde a campos vacíos.

En consecuencia, únicamente se identifican tres proveedores con movimientos registrados dentro de la información analizada. Esta situación resulta relevante debido a que la documentación del negocio menciona la existencia de trece proveedores activos, aspecto que será revisado posteriormente en la sección de calidad de datos.

### Productos

La tabla contiene movimientos asociados a 227 productos diferentes. Este resultado muestra que la información cubre una porción significativa del inventario administrado por la organización y permite realizar análisis sobre una amplia variedad de referencias.

La cantidad de productos observada también coincide con los rangos identificados durante la exploración de la variable `ProductoID`.

### Tipos de transacción

Se encontraron únicamente tres tipos de transacción distintos dentro del conjunto de datos.

Aunque no se dispone de la descripción funcional de cada uno de estos códigos, la baja cardinalidad observada sugiere que los movimientos de inventario son generados por un número limitado de procesos operativos. Posteriormente se analizará la distribución de registros por tipo de transacción para comprender qué tan representativo es cada proceso dentro de la operación.

### Conclusión

La información presenta una diversidad considerable en términos de clientes y productos, mientras que la cantidad de tipos de transacción y proveedores observados es relativamente reducida. Este comportamiento sugiere que la operación involucra una amplia variedad de artículos y clientes, pero se apoya en un conjunto limitado de procesos y entidades asociadas a los movimientos registrados.


In [11]:
movimientos.select("Cantidad").summary().show()

+-------+-----------------+
|summary|         Cantidad|
+-------+-----------------+
|  count|           204292|
|   mean|719.4997650421946|
| stddev| 4729.36659747762|
|    min|           -360.0|
|    25%|            -60.0|
|    50%|             -9.0|
|    75%|             -5.0|
|    max|          67368.0|
+-------+-----------------+



## Distribución de la variable Cantidad

La columna `Cantidad` refleja el número de unidades que ingresan o salen del inventario en cada movimiento registrado. Debido a que esta variable representa directamente el impacto de una transacción sobre las existencias, resulta importante analizar cómo se comportan sus valores dentro del conjunto de datos.

Las principales métricas obtenidas fueron las siguientes:

| Medida                 |  Valor |
| ---------------------- | -----: |
| Mínimo                 |   -360 |
| Percentil 25           |    -60 |
| Mediana (Percentil 50) |     -9 |
| Percentil 75           |     -5 |
| Máximo                 | 67.368 |
| Promedio               |    719 |
| Desviación estándar    |  4.729 |

Al observar estos resultados, se evidencia que la mayoría de los movimientos involucran cantidades relativamente pequeñas. De hecho, el 75% de los registros presenta valores iguales o inferiores a -5 unidades, mientras que la mitad de los movimientos se concentra alrededor de -9 unidades.

Sin embargo, también existen algunas transacciones con cantidades significativamente superiores al comportamiento habitual. Esto se refleja en la diferencia entre el valor máximo registrado (67.368 unidades) y el resto de los percentiles observados.

Otro aspecto relevante es la diferencia entre el promedio (719) y la mediana (-9). Si los datos estuvieran distribuidos de manera uniforme, ambos indicadores tendrían valores más cercanos. En este caso ocurre lo contrario, lo que sugiere que un número reducido de movimientos con cantidades muy elevadas está influyendo de manera importante sobre el promedio general.

En términos prácticos, esto indica que la operación diaria parece estar compuesta principalmente por movimientos pequeños o moderados, mientras que ocasionalmente se registran transacciones de magnitud considerable que modifican el comportamiento global de la variable.

### Conclusión

La variable `Cantidad` no presenta una distribución homogénea. La mayoría de los movimientos se concentra en rangos relativamente bajos, pero existen registros con cantidades considerablemente mayores que generan diferencias importantes entre las medidas estadísticas analizadas. Este comportamiento deberá tenerse en cuenta durante la interpretación de indicadores relacionados con el volumen de inventario movilizado.


In [12]:
movimientos.groupBy("TipoTransaccionID").count().show()

+-----------------+------+
|TipoTransaccionID| count|
+-----------------+------+
|               12|    46|
|               10|197136|
|               11|  7110|
+-----------------+------+



## Distribución de los tipos de transacción

Para entender qué procesos generan los movimientos registrados en la tabla, se analizó la cantidad de registros asociada a cada valor de `TipoTransaccionID`.

| TipoTransaccionID | Cantidad de movimientos | Participación |
| ----------------- | ----------------------: | ------------: |
| 10                |                 197.136 |        96,50% |
| 11                |                   7.110 |         3,48% |
| 12                |                      46 |         0,02% |

La distribución obtenida muestra un comportamiento altamente concentrado. Prácticamente todos los movimientos almacenados en la tabla corresponden al tipo de transacción 10, mientras que los demás tipos tienen una participación muy reducida dentro del total de registros.

En particular, por cada movimiento asociado a los tipos 11 o 12 existen decenas de movimientos pertenecientes al tipo 10. Esto indica que la mayor parte de la actividad registrada durante el período analizado está relacionada con un mismo proceso operativo o evento de negocio.

También resulta llamativo el bajo volumen asociado al tipo de transacción 12, ya que únicamente se identificaron 46 registros en todo el conjunto de datos. Debido a su escasa frecuencia, este tipo de movimiento podría tener un comportamiento particular o corresponder a situaciones excepcionales dentro de la operación.

Dado que la documentación suministrada no incluye la descripción funcional de cada código, no es posible determinar con certeza qué representa cada tipo de transacción. Sin embargo, la diferencia observada en la cantidad de registros sugiere que no todos los procesos tienen la misma relevancia o frecuencia dentro de la gestión de inventarios.

### Conclusión

Los movimientos de inventario analizados se encuentran dominados por un único tipo de transacción, que concentra más del 96% de los registros disponibles. Antes de desarrollar análisis más avanzados, sería conveniente validar con el negocio el significado de cada código para comprender qué procesos están representados y si esta distribución corresponde al comportamiento esperado de la organización.


# Conclusión General del Perfilamiento de Datos

El análisis exploratorio realizado sobre la tabla `movimientosCopia` permitió obtener una visión inicial de la información disponible y del comportamiento general de los movimientos de inventario registrados por Wide World Importers.

A partir de los 204.292 registros analizados se observó que la tabla contiene información suficiente para estudiar la dinámica de inventarios, ya que incorpora datos relacionados con productos, clientes, proveedores, fechas, documentos de soporte y cantidades movilizadas. Esta combinación de atributos permite realizar análisis tanto operativos como históricos sobre los movimientos registrados.

Durante la exploración se identificaron 227 productos diferentes y 664 clientes asociados a los movimientos, lo que evidencia una operación con una participación amplia de artículos y actores comerciales. También se observó que la información cubre transacciones realizadas entre finales de 2013 y septiembre de 2015, proporcionando un horizonte temporal adecuado para analizar tendencias y patrones de comportamiento.

Uno de los hallazgos más relevantes fue la fuerte concentración de registros en el tipo de transacción 10, que representa prácticamente la totalidad de los movimientos almacenados. Este resultado sugiere que la actividad registrada se encuentra dominada por un proceso operativo específico, cuyo significado deberá validarse con las áreas de negocio para comprender correctamente su impacto dentro de la gestión de inventarios.

Respecto a las cantidades movilizadas, se encontró un comportamiento poco uniforme. La mayoría de los movimientos corresponde a volúmenes relativamente bajos, pero también existen algunas transacciones de magnitud considerable que influyen de manera importante en los indicadores estadísticos generales. Este aspecto deberá tenerse presente al momento de interpretar métricas relacionadas con consumo, abastecimiento o rotación de inventarios.

Finalmente, el perfilamiento permitió identificar varios aspectos que merecen una revisión más detallada durante la etapa de calidad de datos, particularmente en lo relacionado con fechas, proveedores, documentos asociados y reglas de negocio. Por esta razón, la siguiente fase del análisis se enfocará en validar la consistencia, completitud y confiabilidad de la información antes de utilizarla para fines analíticos.


### Análisis de calidad de datos

In [13]:
from pyspark.sql.functions import countDistinct

movimientos.select(
    countDistinct("TransaccionProductoID").alias("IdsUnicos")
).show()

print("Total registros:", movimientos.count())

+---------+
|IdsUnicos|
+---------+
|   173659|
+---------+



Total registros: 204292


In [14]:
duplicados = movimientos.count() - movimientos.dropDuplicates().count()

print("Registros duplicados:", duplicados)

Registros duplicados: 30633


# Calidad de Datos

## Dimensión: Unicidad

### Objetivo

Verificar si cada movimiento de inventario se encuentra representado una única vez dentro de la tabla y determinar si existen registros repetidos que puedan afectar la confiabilidad de los análisis posteriores.

### Validación realizada

Para evaluar la unicidad de la información se comparó el número total de registros almacenados con la cantidad de valores distintos presentes en el campo `TransaccionProductoID`, el cual aparentemente corresponde al identificador de cada movimiento.

| Métrica                   |   Valor |
| ------------------------- | ------: |
| Registros totales         | 204.292 |
| Identificadores distintos | 173.659 |
| Diferencia                |  30.633 |

Adicionalmente, se ejecutó una validación sobre registros duplicados utilizando las funcionalidades nativas de PySpark.

| Métrica                            |  Valor |
| ---------------------------------- | -----: |
| Registros duplicados identificados | 30.633 |

### Hallazgo

Los resultados muestran que una parte importante de los registros no es única dentro del conjunto de datos. En total se identificaron 30.633 observaciones duplicadas, cifra que representa aproximadamente el 15% de la información analizada.

También se observó que la cantidad de identificadores distintos es considerablemente menor al número total de registros, lo que indica que el campo `TransaccionProductoID` no está funcionando como una llave única para todos los movimientos almacenados.

### Impacto para el negocio

La presencia de duplicados puede afectar directamente los resultados de cualquier análisis construido sobre esta información. Entre los posibles efectos se encuentran:

* Incremento artificial de los movimientos de inventario reportados.
* Sobreestimación de cantidades entrantes o salientes.
* Distorsión de indicadores operativos y logísticos.
* Diferencias entre reportes analíticos y cifras operativas reales.
* Dificultades durante procesos de conciliación y seguimiento histórico.

Si estos registros duplicados no son tratados adecuadamente, los resultados obtenidos podrían no reflejar el comportamiento real de la operación.

### Posibles alternativas de tratamiento

Antes de eliminar registros es importante comprender el origen de la duplicidad observada. Algunas acciones recomendadas son:

* Revisar si los registros repetidos corresponden a errores de carga o a eventos válidos del negocio.
* Analizar si un mismo identificador puede estar asociado legítimamente a más de un movimiento.
* Definir una llave de negocio que permita identificar de manera inequívoca cada transacción.
* Implementar controles de validación durante los procesos de integración de datos para evitar la generación de nuevos duplicados.

### Conclusión

La revisión realizada evidencia que la unicidad de la información no se encuentra completamente garantizada. Debido al volumen de registros duplicados identificado, resulta recomendable validar su origen antes de utilizar la tabla como fuente para indicadores, reportes o procesos analíticos, ya que estos registros podrían alterar significativamente los resultados obtenidos.


In [15]:
from pyspark.sql.functions import col

print("ClienteID = 0:", movimientos.filter(col("ClienteID") == 0).count())

print("InvoiceID = 0:", movimientos.filter(col("InvoiceID") == 0).count())

print("ProveedorID vacío:", movimientos.filter(col("ProveedorID") == "").count())

print("OrdenDeCompraID vacío:", movimientos.filter(col("OrdenDeCompraID") == "").count())

ClienteID = 0: 7156


InvoiceID = 0: 7156


ProveedorID vacío: 197182


OrdenDeCompraID vacío: 197182


## Dimensión: Completitud

### Objetivo

Evaluar si los atributos disponibles contienen la información necesaria para interpretar adecuadamente los movimientos de inventario y determinar si existen campos que, aun sin presentar valores nulos, puedan estar almacenando información incompleta.

### Validación realizada

Como primer paso se verificó la presencia de valores nulos en las columnas de la tabla. La revisión no evidenció registros nulos en los atributos analizados.

Posteriormente se realizó una búsqueda de valores especiales que podrían estar siendo utilizados para representar ausencia de información, particularmente identificadores con valor cero y campos de texto vacíos.

Los resultados obtenidos fueron los siguientes:

| Campo                 | Registros identificados |
| --------------------- | ----------------------: |
| ClienteID = 0         |                   7.156 |
| InvoiceID = 0         |                   7.156 |
| ProveedorID vacío     |                 197.182 |
| OrdenDeCompraID vacío |                 197.182 |

### Hallazgo

Aunque la tabla no contiene valores nulos explícitos, se observó una cantidad importante de registros que podrían no aportar información útil para determinados análisis.

Los casos más relevantes corresponden a los campos `ProveedorID` y `OrdenDeCompraID`, donde más de 197 mil movimientos no presentan ningún valor registrado. Adicionalmente, se encontraron miles de registros con identificadores de cliente y factura iguales a cero.

A partir de la información disponible no es posible determinar si estos valores representan una condición válida del negocio o si corresponden a datos faltantes almacenados mediante convenciones del sistema.

### Impacto para el negocio

Si estos registros efectivamente representan información ausente, podrían existir limitaciones para responder preguntas relacionadas con el abastecimiento, la trazabilidad documental o la relación entre movimientos y entidades involucradas en la operación.

Por ejemplo, la ausencia de proveedor dificultaría analizar patrones de compra o dependencia de determinados abastecedores, mientras que la falta de órdenes de compra limitaría la reconstrucción completa de algunos procesos operativos.

Sin embargo, también es posible que ciertos tipos de movimiento no requieran registrar esta información, por lo que cualquier conclusión deberá ser validada con los responsables del proceso.

### Aspectos que requieren aclaración

Durante la revisión surgieron varias preguntas cuya respuesta es necesaria para interpretar correctamente los resultados:

1. ¿El valor 0 en `ClienteID` corresponde a una categoría válida dentro del sistema?
2. ¿El valor 0 en `InvoiceID` representa movimientos sin factura asociada o una condición especial de negocio?
3. ¿Existen tipos de transacción que no requieren proveedor?
4. ¿Las órdenes de compra son obligatorias para todos los movimientos o únicamente para ciertos procesos?
5. ¿Las reglas de diligenciamiento cambian dependiendo del `TipoTransaccionID`?

### Posibles acciones de tratamiento

Dependiendo de las respuestas obtenidas por parte del negocio, podrían considerarse las siguientes alternativas:

* Transformar valores cero en nulos cuando representen ausencia de información.
* Definir reglas de obligatoriedad por tipo de transacción.
* Implementar validaciones durante la captura o integración de datos.
* Generar alertas cuando se registren movimientos sin la información requerida para su proceso correspondiente.

### Conclusión

Desde una perspectiva técnica, la tabla se encuentra completamente poblada porque no se identificaron valores nulos. Sin embargo, la existencia de identificadores con valor cero y una elevada cantidad de campos vacíos sugiere que parte de la información podría requerir una revisión funcional. Antes de utilizar estos atributos para construir indicadores o realizar análisis de negocio, resulta conveniente validar el significado operativo de dichos valores con los usuarios responsables de la información.


In [16]:
movimientos.filter(col("Cantidad") > 50000000).count()

0

In [17]:
movimientos.selectExpr(
    "max(Cantidad) as CantidadMaxima"
).show()

+--------------+
|CantidadMaxima|
+--------------+
|       67368.0|
+--------------+



## Dimensión: Validez

### Objetivo

Comprobar si los movimientos almacenados en la tabla respetan una de las restricciones operativas comunicadas por Wide World Importers durante la definición del caso de negocio.

La organización indicó que:

> Ningún movimiento de inventario debería superar los 50 millones de unidades en una sola transacción.

### Validación realizada

Para verificar esta condición se revisó la columna `Cantidad`, identificando el valor máximo registrado y buscando posibles movimientos que excedieran el límite establecido por el negocio.

Los resultados obtenidos fueron los siguientes:

| Métrica                              | Resultado |
| ------------------------------------ | --------: |
| Movimientos superiores a 50 millones |         0 |
| Cantidad máxima observada            |    67.368 |

### Hallazgo

La revisión no evidenció movimientos que sobrepasen el límite definido por la organización.

El valor más alto encontrado corresponde a 67.368 unidades, cifra considerablemente inferior al umbral de 50 millones mencionado en los requerimientos. Esto indica que, al menos para esta regla específica, los datos se comportan de acuerdo con lo esperado.

### Impacto para el negocio

La ausencia de cantidades excesivamente altas reduce la probabilidad de que existan errores evidentes de captura, problemas de integración entre sistemas o registros generados por fallas durante procesos de carga de información.

Adicionalmente, permite tener mayor confianza en los análisis relacionados con consumo de inventario, movimientos de productos y volumen de operaciones, ya que no se identificaron valores extremos que puedan alterar significativamente los resultados.

### Observación

Aunque la regla validada se cumple, esto no garantiza por sí solo que todos los movimientos sean correctos. La validez de la información también depende de otras condiciones de negocio que deberán revisarse en aspectos como fechas, proveedores, documentos asociados y tipos de transacción.

### Conclusión

Con base en la información analizada, no se encontraron evidencias de movimientos que incumplan el límite máximo definido por la organización. Los valores observados se encuentran dentro de los rangos esperados y no se identificaron registros que requieran corrección por este criterio específico de validación.


In [18]:
from pyspark.sql.functions import to_timestamp, col

movimientos_fecha = movimientos.withColumn(
    "FechaConvertida",
    to_timestamp(col("FechaTransaccion"), "yyyy-MM-dd HH:mm:ss.SSSSSSS")
)

movimientos_fecha.filter(
    col("FechaConvertida").isNull()
).count()

64254

In [19]:
movimientos.select("FechaTransaccion").show(10, False)

+----------------+
|FechaTransaccion|
+----------------+
|Apr 25,2014     |
|Dec 10,2015     |
|Dec 04,2015     |
|Dec 23,2015     |
|Jul 27,2015     |
|Sep 15,2014     |
|Aug 04,2015     |
|Feb 23,2015     |
|May 01,2015     |
|Jan 08,2016     |
+----------------+
only showing top 10 rows



In [20]:
movimientos_fecha.select("FechaTransaccion", "FechaConvertida").show(10, False)

+----------------+---------------+
|FechaTransaccion|FechaConvertida|
+----------------+---------------+
|Apr 25,2014     |NULL           |
|Dec 10,2015     |NULL           |
|Dec 04,2015     |NULL           |
|Dec 23,2015     |NULL           |
|Jul 27,2015     |NULL           |
|Sep 15,2014     |NULL           |
|Aug 04,2015     |NULL           |
|Feb 23,2015     |NULL           |
|May 01,2015     |NULL           |
|Jan 08,2016     |NULL           |
+----------------+---------------+
only showing top 10 rows



## Dimensión: Consistencia y validez de las fechas

### Objetivo

Verificar si la información almacenada en el campo `FechaTransaccion` utiliza un formato coherente y compatible con la especificación entregada por Wide World Importers.

Según la documentación suministrada, las fechas deberían registrarse bajo el formato:

```text
YYYY-MM-DD HH:MM:SS
```

### Validación realizada

Para comprobar esta condición se intentó convertir la columna `FechaTransaccion` a un tipo de dato temporal utilizando el formato definido por el negocio.

Durante la revisión se observaron registros con valores como:

* Apr 25,2014
* Dec 10,2015
* Jul 27,2015
* Sep 15,2014

Posteriormente se aplicó una conversión utilizando el formato esperado y se identificaron 64.254 registros que no pudieron interpretarse correctamente como fechas válidas.

### Hallazgo

Los resultados muestran que la información almacenada en `FechaTransaccion` no sigue un único patrón de representación.

Mientras que la documentación indica que las fechas deberían encontrarse en formato numérico con año, mes, día y hora, una cantidad importante de registros utiliza abreviaturas textuales para representar el mes.

Algunos ejemplos observados fueron:

```text
Apr 25,2014
Dec 10,2015
Jul 27,2015
```

Debido a esta diferencia de formatos, una parte significativa de los registros no puede convertirse automáticamente utilizando la regla proporcionada por el negocio.

### Impacto para el negocio

Las fechas son uno de los atributos más importantes para realizar análisis históricos y de tendencias. Cuando no existe uniformidad en su formato, pueden aparecer problemas en distintas etapas del procesamiento de información, por ejemplo:

* Dificultades para ordenar cronológicamente los movimientos.
* Errores durante procesos ETL o cargas automatizadas.
* Problemas al calcular indicadores por período.
* Inconsistencias en reportes históricos.
* Fallas de integración con aplicaciones que esperan un formato específico.

En escenarios analíticos, este tipo de inconsistencias puede afectar directamente la confiabilidad de los resultados obtenidos.

### Aspectos que requieren validación

A partir de los resultados observados surgen varias preguntas que deberían resolverse con el área responsable de la información:

1. ¿Existe un formato oficial diferente al documentado?
2. ¿La información proviene de múltiples sistemas con convenciones distintas para almacenar fechas?
3. ¿Los registros observados corresponden a datos históricos que fueron migrados desde otra fuente?
4. ¿La mezcla de formatos es esperada o representa una situación que debería corregirse?

### Posibles acciones de tratamiento

Para garantizar la consistencia temporal de la información podrían considerarse las siguientes alternativas:

* Convertir todos los registros a un único formato estándar antes de iniciar los análisis.
* Definir reglas de transformación para los formatos textuales identificados.
* Incorporar validaciones durante los procesos de carga para evitar la aparición de nuevos formatos.
* Almacenar las fechas utilizando tipos de dato temporales en lugar de cadenas de texto cuando sea posible.

### Conclusión

La revisión realizada evidencia que la columna `FechaTransaccion` presenta diferencias entre el formato documentado y el formato realmente almacenado en una parte importante de los registros. Debido a que más de 64 mil observaciones no pudieron interpretarse correctamente utilizando la regla definida por el negocio, será necesario normalizar esta información antes de desarrollar análisis temporales o procesos de integración que dependan de la fecha del movimiento.


In [21]:
movimientos.groupBy("ProveedorID").count().show(20, False)

+-----------+------+
|ProveedorID|count |
+-----------+------+
|1.0        |11    |
|4.0        |4832  |
|7.0        |2267  |
|           |197182|
+-----------+------+



## Consistencia frente a la información reportada por el negocio

### Objetivo

Contrastar la información encontrada en la tabla con algunos datos de contexto proporcionados por Wide World Importers, con el fin de identificar posibles diferencias entre la operación descrita por la organización y los registros efectivamente disponibles para análisis.

Dentro de la documentación entregada se indica que la compañía trabaja actualmente con 13 proveedores.

### Validación realizada

Se revisó el contenido del atributo `ProveedorID` y se calculó la cantidad de registros asociada a cada valor encontrado.

Los resultados obtenidos fueron los siguientes:

| ProveedorID | Cantidad de registros |
| ----------- | --------------------: |
| Vacío       |               197.182 |
| 4           |                 4.832 |
| 7           |                 2.267 |
| 1           |                    11 |

Adicionalmente, se contabilizaron los valores distintos presentes en la columna, encontrando únicamente cuatro categorías, una de las cuales corresponde a registros sin información de proveedor.

### Hallazgo

La información observada no coincide completamente con el contexto suministrado por la organización.

Aunque Wide World Importers menciona la existencia de trece proveedores, en la tabla únicamente aparecen movimientos asociados a tres identificadores de proveedor. Además, la gran mayoría de los registros no contiene información en este campo.

En términos porcentuales, aproximadamente el 96,5% de los movimientos analizados no tiene un proveedor asociado dentro de la tabla.

Este resultado puede tener diferentes explicaciones. Por ejemplo, es posible que ciertos movimientos no requieran proveedor, que la información provenga de una muestra parcial de los datos históricos o que existan relaciones almacenadas en otras tablas que no fueron suministradas para esta actividad.

### Impacto para el negocio

La disponibilidad limitada de información sobre proveedores puede restringir algunos análisis relevantes para la gestión de inventarios, entre ellos:

* Seguimiento del comportamiento de los proveedores.
* Evaluación de volúmenes abastecidos por cada proveedor.
* Estudios de dependencia o concentración de abastecimiento.
* Trazabilidad completa de procesos de compra.
* Construcción de indicadores de desempeño relacionados con adquisiciones.

Asimismo, dificulta validar si los movimientos registrados reflejan adecuadamente la totalidad de las relaciones comerciales existentes.

### Aspectos que requieren aclaración

A partir de este resultado sería conveniente validar con el negocio los siguientes puntos:

1. ¿Todos los movimientos deberían tener un proveedor asociado?
2. ¿La información suministrada corresponde a la totalidad de la operación o únicamente a una muestra?
3. ¿Existen tablas complementarias donde se encuentre la relación completa entre productos y proveedores?
4. ¿Los trece proveedores mencionados han tenido movimientos durante el período analizado?
5. ¿El atributo `ProveedorID` se utiliza únicamente para determinados tipos de transacción?

### Posibles acciones de tratamiento

Dependiendo de las respuestas obtenidas, podrían considerarse las siguientes alternativas:

* Incorporar información adicional proveniente de tablas maestras de proveedores.
* Diferenciar los tipos de movimiento que requieren proveedor de aquellos donde este dato no aplica.
* Reemplazar cadenas vacías por valores nulos para facilitar el análisis y la identificación de registros incompletos.
* Definir reglas de validación que permitan controlar la captura de proveedores cuando la operación lo requiera.

### Conclusión

La revisión realizada muestra una diferencia importante entre la información documentada por la organización y los registros observados en la tabla. Dado que únicamente se identificaron movimientos asociados a tres proveedores y que la mayoría de los registros carece de este dato, resulta necesario validar con el negocio cómo debe interpretarse esta situación antes de realizar análisis relacionados con abastecimiento, compras o gestión de proveedores.


In [22]:
236668 - movimientos.count()

32376

## Consistencia frente a los volúmenes reportados por el negocio

### Objetivo

Verificar si la cantidad de movimientos disponible en la tabla analizada coincide con las cifras de referencia entregadas por Wide World Importers.

Dentro de la documentación suministrada se indica que la organización dispone de 236.668 movimientos de inventario registrados desde el año 2013.

### Validación realizada

Para realizar la validación se comparó el número de registros presentes en la tabla `movimientosCopia` con la cantidad de movimientos informada por el negocio.

Los resultados obtenidos fueron los siguientes:

| Métrica                                    |   Valor |
| ------------------------------------------ | ------: |
| Movimientos reportados por la organización | 236.668 |
| Registros encontrados en la tabla          | 204.292 |
| Diferencia                                 |  32.376 |

### Hallazgo

La cantidad de registros disponible para análisis es inferior a la cifra reportada por Wide World Importers.

En términos absolutos se identificó una diferencia de 32.376 movimientos. Esto representa aproximadamente el 13,68% del volumen esperado según la documentación entregada para el proyecto.

A partir de la información disponible no es posible determinar si esta diferencia corresponde a registros faltantes, a una extracción parcial de la información o a una actualización posterior de las cifras de negocio.

### Impacto para el negocio

Cuando existe una diferencia entre los volúmenes reportados y los registros efectivamente disponibles, surge el riesgo de que los análisis no representen completamente la realidad operativa de la organización.

Entre las posibles consecuencias se encuentran:

* Indicadores calculados sobre información incompleta.
* Subestimación del volumen real de movimientos.
* Pérdida de visibilidad sobre determinados períodos o procesos.
* Dificultades para conciliar resultados analíticos con reportes operativos.
* Conclusiones que no reflejen la totalidad de la actividad registrada por la compañía.

Por esta razón, resulta importante comprender el origen de la diferencia antes de utilizar la información como fuente oficial para análisis estratégicos.

### Aspectos que requieren aclaración

Durante la validación surgieron varias preguntas que deberían resolverse con los responsables del negocio y de la plataforma de datos:

1. ¿La tabla `movimientosCopia` contiene todos los movimientos históricos o únicamente una parte de ellos?
2. ¿Se aplicó algún filtro durante la generación del conjunto de datos entregado para la actividad?
3. ¿La cifra de 236.668 movimientos corresponde exactamente al mismo período cubierto por la tabla?
4. ¿Existen movimientos archivados o almacenados en otras fuentes de información?
5. ¿La cifra reportada por la organización fue actualizada después de la extracción de los datos utilizados en esta práctica?

### Posibles acciones de validación

Para determinar el origen de la diferencia identificada podrían realizarse las siguientes actividades:

* Comparar la información con la tabla fuente original.
* Conciliar registros por año y por tipo de transacción.
* Revisar los criterios utilizados durante la extracción de datos.
* Verificar si existen procesos de depuración o archivado que hayan reducido el número de registros disponibles.

### Conclusión

La revisión realizada muestra que el volumen de información contenido en la tabla no coincide completamente con la cifra reportada por la organización. Antes de construir indicadores o generar conclusiones de negocio, sería recomendable validar el origen de esta diferencia para confirmar que la información disponible representa adecuadamente la operación que se desea analizar.


In [23]:
movimientos.groupBy("TipoTransaccionID").agg(
    count("*").alias("Total"),
    count("ProveedorID").alias("ProveedorInformado"),
    count("OrdenDeCompraID").alias("OrdenInformada")
).show()

+-----------------+------+------------------+--------------+
|TipoTransaccionID| Total|ProveedorInformado|OrdenInformada|
+-----------------+------+------------------+--------------+
|               12|    46|                46|            46|
|               10|197136|            197136|        197136|
|               11|  7110|              7110|          7110|
+-----------------+------+------------------+--------------+



In [24]:
movimientos.groupBy("TipoTransaccionID").count().show()

+-----------------+------+
|TipoTransaccionID| count|
+-----------------+------+
|               12|    46|
|               10|197136|
|               11|  7110|
+-----------------+------+



## Análisis de la representatividad de los tipos de transacción

### Objetivo

Examinar cómo se distribuyen los movimientos entre los distintos tipos de transacción presentes en la tabla y determinar si existe algún grupo que concentre la mayor parte de la actividad registrada.

### Validación realizada

Se contabilizó el número de movimientos asociado a cada valor de `TipoTransaccionID` con el fin de conocer la participación relativa de cada categoría dentro del conjunto de datos.

| TipoTransaccionID | Cantidad de registros |
| ----------------- | --------------------: |
| 10                |               197.136 |
| 11                |                 7.110 |
| 12                |                    46 |

### Hallazgo

La distribución observada muestra una concentración muy marcada en el tipo de transacción 10.

De los 204.292 movimientos analizados, 197.136 pertenecen a esta categoría, lo que equivale aproximadamente al 96,5% de toda la información disponible. En contraste, el tipo 11 representa una fracción mucho menor de los registros y el tipo 12 aparece únicamente en 46 ocasiones durante todo el período analizado.

Esta diferencia sugiere que los distintos procesos asociados a movimientos de inventario no tienen la misma frecuencia dentro de la operación o que algunos eventos son considerablemente menos comunes que otros.

### Impacto para el negocio

La concentración observada puede influir directamente en la interpretación de los resultados analíticos.

Por ejemplo:

* Los indicadores globales estarán fuertemente influenciados por el comportamiento del tipo de transacción 10.
* Las conclusiones obtenidas para la totalidad de la tabla podrían no reflejar adecuadamente el comportamiento de los tipos 11 y 12.
* Los análisis comparativos entre tipos de transacción pueden verse limitados debido al reducido número de observaciones disponibles para algunas categorías.
* Los modelos estadísticos o predictivos podrían aprender principalmente patrones asociados al tipo de transacción dominante.

Por esta razón, resulta importante entender el significado operativo de cada categoría antes de realizar interpretaciones de negocio.

### Aspectos que requieren aclaración

Durante la revisión surgieron varias preguntas que sería conveniente validar con los responsables del proceso:

1. ¿Qué actividad o proceso representa cada valor de `TipoTransaccionID`?
2. ¿La distribución observada es consistente con la operación habitual de la compañía?
3. ¿Existen tipos de transacción adicionales que no aparecen en la información entregada?
4. ¿Cada tipo de transacción tiene reglas distintas respecto a clientes, proveedores o documentos asociados?
5. ¿El bajo volumen del tipo 12 corresponde a eventos excepcionales o a una posible ausencia de información?

### Consideraciones para el análisis

Antes de construir indicadores o realizar comparaciones entre categorías, sería recomendable:

* Analizar cada tipo de transacción por separado.
* Validar si las diferencias observadas corresponden al comportamiento esperado del negocio.
* Considerar la baja representatividad de algunas categorías al interpretar resultados estadísticos.
* Confirmar con el negocio el significado funcional de cada código.

### Conclusión

Los movimientos registrados no se distribuyen de manera uniforme entre los tipos de transacción disponibles. La información se encuentra dominada por una única categoría que concentra prácticamente toda la actividad observada. Por esta razón, cualquier análisis posterior deberá considerar esta concentración para evitar interpretaciones que generalicen comportamientos que realmente pertenecen a un solo tipo de proceso operativo.


In [25]:
movimientos.filter(col("Cantidad") < 0).count()

197158

In [26]:
movimientos.filter(col("Cantidad") < 0)\
.select("TipoTransaccionID","Cantidad")\
.show(20, False)

+-----------------+--------+
|TipoTransaccionID|Cantidad|
+-----------------+--------+
|10               |-40.0   |
|10               |-7.0    |
|10               |-2.0    |
|10               |-3.0    |
|10               |-24.0   |
|10               |-20.0   |
|10               |-60.0   |
|10               |-3.0    |
|10               |-7.0    |
|10               |-216.0  |
|10               |-20.0   |
|10               |-25.0   |
|10               |-36.0   |
|10               |-8.0    |
|10               |-4.0    |
|10               |-96.0   |
|10               |-260.0  |
|10               |-24.0   |
|10               |-10.0   |
|10               |-6.0    |
+-----------------+--------+
only showing top 20 rows



## Validación del comportamiento de la variable Cantidad

### Objetivo

Analizar si los valores registrados en la columna `Cantidad` son consistentes con la lógica esperada de los movimientos de inventario y determinar si existen patrones que requieran validación adicional por parte del negocio.

### Validación realizada

Se examinó la distribución de los valores de la variable `Cantidad`, prestando especial atención a la presencia de movimientos con cantidades negativas.

Los resultados obtenidos fueron los siguientes:

| Métrica                         |   Valor |
| ------------------------------- | ------: |
| Registros totales               | 204.292 |
| Registros con cantidad negativa | 197.158 |
| Participación                   |  96,51% |

Durante la revisión también se observó que la mayoría de estos registros se encuentra asociada al `TipoTransaccionID` 10.

Algunos ejemplos identificados dentro de los datos incluyen valores como:

* -2
* -3
* -7
* -20
* -40
* -216

### Hallazgo

La característica más llamativa de esta variable es que casi la totalidad de los movimientos presenta cantidades negativas.

Desde una perspectiva puramente estadística, este comportamiento podría parecer inusual. Sin embargo, en sistemas de inventario es frecuente utilizar convenciones de signo para diferenciar entradas y salidas de productos. Bajo esta lógica, una cantidad negativa podría representar una disminución de existencias y una cantidad positiva un incremento del inventario.

Debido a que la documentación suministrada no describe el significado de cada tipo de transacción, no es posible determinar únicamente a partir de los datos si este comportamiento corresponde a una regla operativa válida o a una situación que requiere corrección.

### Impacto para el negocio

La correcta interpretación de esta variable es fundamental para cualquier análisis relacionado con inventarios.

Una comprensión incorrecta del significado de los signos podría afectar:

* El cálculo de consumo de productos.
* Los indicadores de rotación de inventario.
* Las estimaciones de abastecimiento.
* Los análisis de entradas y salidas de mercancía.
* La interpretación de tendencias de inventario a lo largo del tiempo.

Por esta razón, resulta indispensable validar la semántica de los movimientos antes de construir indicadores de negocio.

### Aspectos que requieren aclaración

Para interpretar adecuadamente la información sería conveniente validar con el negocio los siguientes puntos:

1. ¿Las cantidades negativas representan salidas de inventario?
2. ¿Las cantidades positivas representan entradas de inventario?
3. ¿Cada tipo de transacción utiliza la misma convención de signos?
4. ¿Existen excepciones donde una cantidad negativa tenga un significado diferente?
5. ¿Cuál es la relación entre el signo de la cantidad y los códigos de `TipoTransaccionID`?

### Consideraciones para el tratamiento de datos

Antes de realizar transformaciones sobre esta variable se recomienda:

* Confirmar las reglas operativas asociadas a los signos.
* Documentar el significado de cada tipo de movimiento.
* Evitar convertir cantidades negativas en positivas sin validación previa.
* Incorporar reglas de negocio que permitan interpretar correctamente los movimientos de inventario.

### Conclusión

La elevada proporción de cantidades negativas constituye una característica relevante del conjunto de datos, pero no puede considerarse automáticamente un problema de calidad. Con la información disponible, la explicación más probable es que estas cantidades representen salidas de inventario. No obstante, será necesario confirmar esta interpretación con los responsables del proceso para garantizar que los análisis posteriores reflejen correctamente la operación de la organización.


### Conclusiones

# Conclusiones para el negocio

El análisis realizado sobre la tabla `movimientosCopia` permitió obtener una visión general de los movimientos de inventario registrados por Wide World Importers y evaluar si la información disponible es suficiente para soportar futuras iniciativas analíticas.

Desde una perspectiva de cobertura, los resultados son positivos. La tabla contiene 204.292 movimientos de inventario registrados entre 2013 y 2015, asociados a 227 productos diferentes y 664 clientes. Este volumen de información constituye una base adecuada para desarrollar análisis descriptivos, construir indicadores operativos y comprender el comportamiento histórico de los movimientos de inventario.

Sin embargo, durante el proceso de perfilamiento y validación se identificaron varios aspectos que deberían revisarse antes de utilizar la información como fuente oficial para la toma de decisiones.

El primer hallazgo relevante corresponde a la presencia de 30.633 registros duplicados. Debido a que estos registros representan una proporción significativa del conjunto de datos, podrían generar sobreestimaciones en indicadores relacionados con movimientos, consumo de inventario y volumen de operaciones.

También se observó una diferencia entre la cantidad de movimientos reportada por la organización y la cantidad de registros disponibles en la tabla analizada. Mientras la documentación menciona 236.668 movimientos, la información suministrada contiene 204.292 registros. Esta diferencia requiere validación para determinar si corresponde a una extracción parcial de datos, a procesos de depuración previos o a diferencias en los períodos analizados.

Otro aspecto importante está relacionado con la información temporal. Se identificaron distintos formatos dentro del campo `FechaTransaccion`, situación que dificulta la conversión automática de las fechas y podría afectar análisis históricos, reportes periódicos e integraciones con otras fuentes de información.

Adicionalmente, se encontró una disponibilidad limitada de información relacionada con proveedores y órdenes de compra. Más del 96% de los movimientos no presenta valores registrados en estos atributos, lo que restringe la posibilidad de realizar análisis detallados sobre abastecimiento, compras y desempeño de proveedores.

Durante la revisión también se detectó una diferencia entre la información reportada por la organización y los registros observados respecto a los proveedores. Aunque la documentación menciona trece proveedores, únicamente se identificaron movimientos asociados a tres proveedores dentro de la tabla analizada. Será necesario validar si esta situación corresponde a una característica propia del proceso o a una limitación de la información suministrada.

Finalmente, se observó que la mayoría de los movimientos presenta cantidades negativas. Aunque este comportamiento podría ser completamente válido dentro de la lógica de inventarios, resulta indispensable confirmar con el negocio el significado de cada tipo de transacción para garantizar una interpretación correcta de los datos.

En términos generales, la información disponible constituye una base valiosa para desarrollar la solución analítica planteada por Wide World Importers. No obstante, antes de avanzar hacia etapas de reportería, construcción de indicadores o modelamiento, se recomienda realizar un proceso de validación funcional con las áreas responsables de negocio y ejecutar actividades de depuración orientadas a resolver las inconsistencias identificadas.

Con base en los resultados obtenidos, se concluye que los datos son utilizables para fines analíticos, pero requieren algunas verificaciones adicionales para garantizar que las conclusiones derivadas reflejen con precisión la realidad operativa de la organización.
